# <font color="red"> ==============generating the seeds for dSTR from connectivity density map==================== </font>

In [2]:
import os

import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import pandas as pd
import sys
sys.path.append("/Users/aeed/Dropbox/SCRIPTS/ABA_connectivty_atlas")
from structural_conn_functions import create_293_ROI_atlas, create_ROI_293_bilateral_atlas, extract_tile, extract_top_projection_structures, generate_structural_connectivity_matrix, generate_top_projection_structures_atlas, get_structures_from_mask, is_experiment_in_right_hemisphere


In [3]:
working_dir = "/Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/STRd_seed_based_analysis"
os.makedirs(working_dir, exist_ok=True)

In [4]:
avg_STRd_density_map = "/Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps/final_avg.nii.gz"

In [7]:
# genertae 51 ROIs bilateral atlas with 100um resolution to extract the STRd region
out_atlas = create_293_ROI_atlas(structure_set_id=3,
                                 resolution=0.1,
                                 output_dir=working_dir)

2026-05-04 10:28:33,614 allensdk.api.api.retrieve_file_over_http INFO     Downloading URL: http://download.alleninstitute.org/informatics-archive/current-release/mouse_ccf/average_template/average_template_100.nrrd
2026-05-04 10:28:34,203 allensdk.api.api.retrieve_file_over_http INFO     Downloading URL: http://download.alleninstitute.org/informatics-archive/current-release/mouse_ccf/annotation/ccf_2017/structure_masks/structure_masks_100/structure_184.nrrd
2026-05-04 10:28:34,475 allensdk.api.api.retrieve_file_over_http INFO     Downloading URL: http://download.alleninstitute.org/informatics-archive/current-release/mouse_ccf/annotation/ccf_2017/structure_masks/structure_masks_100/structure_500.nrrd
2026-05-04 10:28:34,752 allensdk.api.api.retrieve_file_over_http INFO     Downloading URL: http://download.alleninstitute.org/informatics-archive/current-release/mouse_ccf/annotation/ccf_2017/structure_masks/structure_masks_100/structure_453.nrrd
2026-05-04 10:28:35,038 allensdk.api.api.ret

Wrote atlas: /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/STRd_seed_based_analysis/Atlas_res_100_set_id_3_ROIs_51.nii.gz
Wrote labels: /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/STRd_seed_based_analysis/Atlas_res_100_set_id_3_ROIs_51_labels.csv
Wrote colormap: /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/STRd_seed_based_analysis/Atlas_res_100_set_id_3_ROIs_51.cmap


In [8]:
atlas_path = f"{working_dir}/Atlas_res_100_set_id_3_ROIs_51.nii.gz"
labels_path = f"{working_dir}/Atlas_res_100_set_id_3_ROIs_51_labels.csv"

out_bilateral_atlas = create_ROI_293_bilateral_atlas(
    atlas_path=atlas_path,
    labels_path=labels_path,

)

Number of ROIs in the atlas: 51
Wrote bilateral atlas: /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/STRd_seed_based_analysis/Atlas_res_100_set_id_3_ROIs_51_bilateral.nii.gz
Wrote bilateral labels: /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/STRd_seed_based_analysis/Atlas_res_100_set_id_3_ROIs_51_bilateral_labels.csv
Wrote lut file: None
Wrote bilateral colormap: /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/STRd_seed_based_analysis/Atlas_res_100_set_id_3_ROIs_51_bilateral.cmap


In [11]:
# delete the intermediate data
!imrm {working_dir}/structure_mask_*.nii.gz
!rm -rf {working_dir}/annotation

In [12]:
bilateral_51_atlas = f"{working_dir}/Atlas_res_100_set_id_3_ROIs_51_bilateral.nii.gz"
bilateral_51_atlas_labels = f"{working_dir}/Atlas_res_100_set_id_3_ROIs_51_bilateral_labels.csv"

In [13]:
# extract the label value for dSTR from the labels
labels_df = pd.read_csv(bilateral_51_atlas_labels)
STRd_label_value = labels_df[labels_df["acronym"] == "R_STRd"]["label_value"].values[0]
print(f"Right Dorsal STR label value: {STRd_label_value}")

Right Dorsal STR label value: 25


In [19]:
# extract the STRd region from the bilateral atlas
atlas_img = nib.load(bilateral_51_atlas)
atlas_data = atlas_img.get_fdata()
STRd_mask = atlas_data == STRd_label_value

# save the STRd seed as a new NIfTI file
STRd_img = nib.Nifti1Image(STRd_mask.astype(np.uint8), atlas_img.affine, atlas_img.header)
STRd_path = os.path.join(working_dir, "STRd.nii.gz")
nib.save(STRd_img, STRd_path)
print(f"STRd seed saved at: {STRd_path}")

STRd seed saved at: /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/STRd_seed_based_analysis/STRd.nii.gz


In [26]:
# now, multiply the STRd mask with the average STRd density map to get a seed
density_img = nib.load(avg_STRd_density_map)
density_data = density_img.get_fdata()
STRd_seed_data = STRd_mask * density_data
# threshold at 0.01 to keep only the voxels with some density
STRd_seed_data[STRd_seed_data < 0.01] = 0
# save the STRd seed as a new NIfTI file
STRd_seed_img = nib.Nifti1Image(STRd_seed_data.astype(np.bool_), atlas_img.affine, atlas_img.header)
STRd_seed_path = os.path.join(working_dir, "STRd_seed.nii.gz")
nib.save(STRd_seed_img, STRd_seed_path)
print(f"STRd seed saved at: {STRd_seed_path}")

STRd seed saved at: /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/STRd_seed_based_analysis/STRd_seed.nii.gz


In [27]:
STRd_mask.shape

(132, 80, 114)

In [28]:
density_data.shape

(132, 80, 114)

### now, trasnform to RABIES space using the same transformation as the atlas

In [29]:
# transform the bilateral atlas
!~/Dropbox/SCRIPTS/ABA2DSURQE/./ABA2DSURQE_via_resampled.sh  {working_dir}/STRd_seed.nii.gz /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/rabies_one_vol.nii.gz   {working_dir}/STRd_seed_RABIES.nii.gz


Using double precision for computations.
Input scalar image: /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/STRd_seed_based_analysis/STRd_seed.nii.gz
Reference image: /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/rabies_one_vol.nii.gz
The composite transform comprises the following transforms (in order): 
  1. /Users/aeed/Documents/Work/RABIES_templates/ABA2DSURQE/transformation_1.h5[0] (type = DisplacementFieldTransform)
  2. /Users/aeed/Documents/Work/RABIES_templates/ABA2DSURQE/transformation_1.h5[1] (type = AffineTransform)
  3. /Users/aeed/Documents/Work/RABIES_templates/ABA2DSURQE/transformation_2.h5[0] (type = AffineTransform)
  4. /Users/aeed/Documents/Work/RABIES_templates/ABA2DSURQE/transformation_3.h5[0] (type = AffineTransform)
  5. /Users/aeed/Documents/Work/RABIES_templates/ABA2DSURQE/transformation_3.h5[1] (type = DisplacementFieldTransform)
Default pixel value: 0
Interpolation type: NearestNeighborInterpolateImageFunction
Output warped image: 

In [5]:
# transform the average STRd density map to RABIES space
!~/Dropbox/SCRIPTS/ABA2DSURQE/./ABA2DSURQE_via_resampled.sh  {avg_STRd_density_map} /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/rabies_one_vol.nii.gz   {working_dir}/avg_STRd_density_map_RABIES.nii.gz

Using double precision for computations.
Input scalar image: /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps/final_avg.nii.gz
Reference image: /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/rabies_one_vol.nii.gz
The composite transform comprises the following transforms (in order): 
  1. /Users/aeed/Documents/Work/RABIES_templates/ABA2DSURQE/transformation_1.h5[0] (type = DisplacementFieldTransform)
  2. /Users/aeed/Documents/Work/RABIES_templates/ABA2DSURQE/transformation_1.h5[1] (type = AffineTransform)
  3. /Users/aeed/Documents/Work/RABIES_templates/ABA2DSURQE/transformation_2.h5[0] (type = AffineTransform)
  4. /Users/aeed/Documents/Work/RABIES_templates/ABA2DSURQE/transformation_3.h5[0] (type = AffineTransform)
  5. /Users/aeed/Documents/Work/RABIES_templates/ABA2DSURQE/transformation_3.h5[1] (type = DisplacementFieldTransform)
Default pixel value: 0
Interpolation type: NearestNeighborInterpolateImageFunction
Out

In [ ]:
# then run rabies with the STRd seed as the input seed to get the dSTR connectivity map